# TP N°4 — SGBD vectoriels avec **Chroma** 

## 🎯 Objectifs d'apprentissage  
Découvrir le flux complet d’une base vectorielle : création d’embeddings, insertion dans un SGBD vectoriel, requêtes de similarité (k‑NN) et mini‑évaluation sur un corpus jouet.  

**Prérequis conseillés**  
- Python 3.10+  
- `pip install chromadb sentence-transformers scikit-learn matplotlib`  




## 🧩 Format du TP
1. Installation & imports  
2. Chargement d’un petit corpus (textes courts)  
3. Génération d’**embeddings** (Sentence‑Transformers MiniLM)  
4. **Chroma** : création d'une collection et  insertion de données  
5. Requêtes Chroma  (Symboliques et vectorielles)
6. Mini‑**évaluation** (précision@k / recall@k sur requêtes annotées)  
7. Visualisation 2D de l’espace vectoriel (PCA)  

> 💡 **Travail à faire** : Exécutez le notebook cellule par cellule et répondez aux questions (compléter le code, explications). Les corrections seront collectives avec partage d'écran pendant les séances. Lisez bien les commentaires et les questions.


## 1- Installation & imports

### ⚙️ Préambule — Environnement (threads / tokenizers) 

In [ ]:
# Limitation du  nombre de threads (OMP/MKL) 
# et désactivation du parallélisme des tokenizers pour éviter 
# que le notebook monopolise la machine 
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
# Si nécessaire, décommentez pour installer (exécuter une seule fois)
# !pip install -U chromadb sentence-transformers scikit-learn matplotlib
# Bonus Qdrant :
# !pip install -U qdrant-client

import os
import json
from typing import List, Dict, Any

import numpy as np
import matplotlib.pyplot as plt

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector DB Chroma
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions 

# Évaluation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Outil pratique pour afficher des structures Python de manière lisible
from pprint import pprint



## 2- Petit corpus jouet

In [ ]:
# Corpus jouet : 20 documents courts (mélange actualités simples et généralités)
docs = [
    "La banque centrale relève ses taux directeurs.",
    "Le chat dort sur le canapé du salon.",
    "L'équipe de France a gagné le match hier soir.",
    "La météo annonce de la pluie demain.",
    "Le conducteur accélère sur l'autoroute A6.",
    "Un félin se repose tranquillement.",
    "Le gouvernement présente un nouveau budget.",
    "Le soleil brille et le ciel est bleu.",
    "La voiture roule vite sur l'autoroute.",
    "Le président visite la capitale.",
    "Le café du coin propose des pâtisseries maison.",
    "Une baisse de l'inflation est attendue.",
    "Le chien court dans le jardin.",
    "La centrale nucléaire est contrôlée régulièrement.",
    "Le professeur corrige les copies des étudiants.",
    "La banque annonce des bénéfices record.",
    "Le train est à l'heure aujourd'hui.",
    "La pluie a cessé et le vent tombe.",
    "Le ministre inaugure un nouveau musée.",
    "Le chat chasse une souris dans la cuisine."
]

# ID et métadonnées associées (ex : catégorie naïve)
ids = [f"doc_{i:03d}" for i in range(len(docs))]
metas = []
for i, t in enumerate(docs):
    if "banque" in t or "taux" in t or "inflation" in t or "budget" in t or "bénéfices" in t:
        cat = "économie"
    elif "chat" in t or "chien" in t or "félin" in t:
        cat = "animaux"
    elif "match" in t or "équipe" in t:
        cat = "sport"
    elif "pluie" in t or "météo" in t or "soleil" in t or "ciel" in t or "vent" in t:
        cat = "météo"
    elif "autoroute" in t or "voiture" in t or "train" in t:
        cat = "transport"
    elif "président" in t or "gouvernement" in t or "ministre" in t:
        cat = "politique"
    else:
        cat = "autre"
    metas.append({"categorie": cat})

len(docs), docs[0], metas[0]


## 3- Génération des embeddings

In [ ]:
# CHOIX d'un encodage externe préalable à l'insertion des documents dans Chroma
# MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

# Encode : embeddings normalisés (L2) pour utiliser cos_sim ≈ produit scalaire
embeddings = model.encode(docs, normalize_embeddings=True)

## 4- Chroma : création d'une collection et insertion  de données


  
### 🎯 **Objectif :** comprendre comment initialiser une base Chroma en mode persistant, créer une collection vectorielle et y insérer des documents

### 🧩 Questions 

1 - La ligne `collection.add(documents=docs, ids=ids, embeddings=embeddings.tolist(), metadatas=metas)` ajoute des éléments dans la collection. Quelles sont les quatre informations transmises ? Que se passerait-il si des identifiants étaient dupliqués ?  
2- La variable `snap = collection.get(include=["metadatas", "documents"])` permet d’inspecter la collection. Quelle est le format de snap` ? Que signifie include ? 
3 - Modifiez le code pour afficher uniquement les textes des documents renvoyés par la requete get.


In [ ]:
# Démarrage Chroma en mode persistant (dossier local 'chroma_tp')
persist_dir = "chroma_tp"
client = chromadb.Client(Settings(is_persistent=True, persist_directory=persist_dir))

## (Ré)initialisation facultative  
# Supprimer toutes les collections
# for c in client.list_collections():
#     client.delete_collection(name=c.name)
# print("Toutes les collections ont été supprimées.")
#Afficher la liste des collections
print(client.list_collections())

In [ ]:

# Création d'une collection (si elle existe déjà, on la récupère)
collection = client.get_or_create_collection(
    name="corpus_demo",
    metadata={"hnsw:space": "ip"}  
)

# Insertion des documents 
collection.add(documents=docs, ids=ids, embeddings=embeddings.tolist(), metadatas=metas)

#CODE ALTERNATIF choix d'encodage dans Chroma
# choix d'une fonction d'embedding
# embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
#     model_name="all-MiniLM-L6-v2")
# collection = client.get_or_create_collection(name="demo_auto",
#    embedding_function=embedder, metadata={"hnsw:space": "ip"} ) # espace de similarité (cosine|l2|ip)
# ajout des documents
# collection.add(
#     documents=docs,
#     ids=ids,
#     metadatas=metas)

print("Taille de la collection :", collection.count())

#Test Récupération de la collection
snap = collection.get(include=["metadatas", "documents"])
print("Clés disponibles :", list(snap.keys()))
print("Nombre de documents indexés :", len(snap["ids"]))




## 5- Requêtes 


  
### 🎯 **Objectif :** comprendre comment rechercher des documents dans une BD chroma. 

### 🧩 Questions 

A - Exécutez le code suivant et expliquez ce que réalise la requête exprimée :  
    1 - Que signifient les différents éléments de la requête : query_texts ? n_results ? include? where?
    2 - Cette requête est-elle une vectorielle "pure" basée uniquement sur des calculs de similarité?   
    3 - La clause where relève-t-elle d'un calcul de similarité?


B - Implémentez les requêtes suivantes :  

 1 - Afficher les documents ayant pour identifiant doc_001 et doc_015 avec leur catégorie.  
 2 - Afficher tous les documents avec categorie = "économie".  
 3 - Afficher les 3 documents les plus proches de la requête « Hausse des taux d’intérêt » en précisant leur texte, leur ids, leurs métadonnées (catégorie) et leur score de similarité avec la requête.   
 4 - Afficher les 3 documents les plus proches de la requête « pluie et vent demain » en précisant leur texte, leur ids, leurs métadonnées (catégorie) et leur score de similarité avec la requête« pluie et vent demain », mais en restreignant  la recherche au sous-corpus categorie = "météo".  
 5 - Définir deux versions de requête qui cherche les 2 documents les plus proches de « La banque centrale annonce » : une version avec query_texts et une version avec query_embeddings puis comparer l’ordre des résultats et les valeurs des scores obtenus.



In [ ]:

res = collection.query(
    query_texts=["chat sur canapé"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)
# Requete de recherche
res = collection.query(
    query_texts=["chat"],     # la requête sert juste à activer la recherche
    n_results=10,             # nombre max de résultats
    where={"categorie": "animaux"}, # filtre sur la métadonnée
    include=["documents", "metadatas"]
)
print("Documents trouvés :")
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(f"- {meta['categorie']:8} | {doc}")
print("Résultats filtrés (categorie='animaux') :", res["ids"][0])


# 1 - Afficher les documents ayant pour identifiant doc_001 et doc_015 avec leur catégorie.
  
# 2 - Afficher tous les documents avec categorie = "économie".
   
#  3 - Afficher les 3 documents les plus proches de la requête « Hausse des taux d’intérêt » en précisant leur texte, 
# leur ids, leurs métadonnées (catégorie) et leur score de similarité avec la requête. 


#  4 - Afficher les 3 documents les plus proches de la requête « pluie et vent demain » 
# en précisant leur texte, leur ids, leurs métadonnées (catégorie) et 
# leur score de similarité avec la requête« pluie et vent demain », mais en restreignant  
# la recherche au sous-corpus categorie = "météo".  

#  5 - Définir deux versions de requête qui cherche les 2 documents les plus proches de « La banque centrale annonce » : 
# une version avec query_texts et 


# une version avec query_embeddings puis 
# comparer l’ordre des résultats et les valeurs des scores obtenus.

## 5- Mini‑évaluation (Precision@k / Recall@k)

### 🎯 **Objectif :** évaluer la qualité du moteur vectoriel à l’aide des métriques de précision et de rappel sur un petit ensemble de requêtes annotées.

### 🧩 Questions 

1 - Ajoutez 5 requêtes annotées supplémentaires dans la variable eval_queries et calculez `P@5` et `R@5`  
👉 Que signifie un P@5 = 1.0 ?  
👉 Pourquoi R@5 dépend-il du nombre total de documents pertinents dans le corpus ?
2 - Expliquez le rôle de chaque variable dans le code ci-dessous.  
3 - Modifiez la valeur de K (essayez 1, 3, 5). Que remarquez-vous ?  
4 - Ajoutez une requête `"Un chien aboie fort"` dans `eval_queries` et observez le résultat.    
5 - Quelle catégorie semble la plus difficile à retrouver ? Pourquoi ?  
6 - Quelle différence voyez-vous entre P@K et R@K dans vos résultats ?



In [ ]:
# Jeu de requêtes annotées (catégorie attendue)
eval_queries = [
    ("Hausse des taux", "économie"),
    ("Le chat se repose", "animaux"),
    ("Il va pleuvoir", "météo"),
    ("Le conducteur roule vite", "transport"),
    ("Le président parle", "politique")
]

K = 2  
scores = []

# On récupère une fois toutes les métadonnées du corpus
metas_all = collection.get(include=["metadatas"])["metadatas"]

for q, gold in eval_queries:
    # 1) Requête vectorielle associée à la requete annotée
    q_emb = model.encode([q], normalize_embeddings=True)
    res = collection.query(query_embeddings=q_emb, n_results=K, include=["metadatas"])

    # 2) Liste des catégories des K premiers documents résultats obtenus
    cats_topk = [m["categorie"] for m in res["metadatas"][0]]

    # 3) Comptages simples
    # nombre de documents dont la catégories est égale à la catégorie attendue (gold)
    # nombre de vrais positifs dans le top-K ()
    nb_positifs = sum(c == gold for c in cats_topk) 
    # nb total de documents du corpus de la catégorie attendue (gold)
    # nb total de documents pertinents dans le corpus
    nb_total_rel = sum(m["categorie"] == gold for m in metas_all)  

    # 4) Mesures
    p = nb_positifs / K if K else 0.0 #Précision @k
    r = nb_positifs / nb_total_rel if nb_total_rel else 0.0 # Rappel - recall @k

    scores.append((q, gold, p, r))

# Affichage propre
print(f"Precision/Recall@{K}")
for q, gold, p, r in scores:
    print(f"- {q:<40} | gold={gold:<10} | P@{K}={p:10.2f} | R@{K}={r:10.2f}")

print(f"\nMoyennes : P@{K}={sum(p for *_, p, _ in scores)/len(scores):.2f} | "
      f"R@{K}={sum(r for *_, _, r in scores)/len(scores):.2f}")


## 6- Visualisation 2D (PCA)

In [ ]:
# PCA des embeddings -> visualisation par catégorie
pca = PCA(n_components=2, random_state=42)
X2 = pca.fit_transform(embeddings)

cats = [m["categorie"] for m in metas]
labels = sorted(set(cats))
label_to_int = {lab: i for i, lab in enumerate(labels)}
colors = [label_to_int[c] for c in cats]

plt.figure(figsize=(6,5))
for lab in labels:
    idx = [i for i, c in enumerate(cats) if c == lab]
    plt.scatter(X2[idx,0], X2[idx,1], label=lab)
plt.legend()
plt.title("Corpus (PCA des embeddings)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()
